# TCN Multi-Output — GWO-Optimized Final Evaluation (H=10)

**Author:** Ibrahim Hanafy  
**Date:** August 2026  

**Best hyperparameters from GWO search (Eval 5, RMSE=0.026152):**

| Parameter | Value | Source |
|-----------|-------|--------|
| `n_filters` | 256 | GWO-optimized |
| `dropout` | 0.109 | GWO-optimized |
| `lr` | 0.002066 | GWO-optimized |
| `n_blocks` | 4 | Fixed |
| `kernel_size` | 3 | Fixed |
| `batch_size` | 128 | Fixed |

**Evaluation:** All 21 patients × H=10, 200 epochs, patience=50

---

## 1 — Imports & Setup

In [1]:
! pip install wfdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 4.0 MB/s eta 0:00:00


In [2]:
import os, time, gc, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wfdb
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

In [3]:
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv1D, Dense, Dropout, Add, Activation, BatchNormalization
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print(f'TensorFlow {tf.__version__}')
print(f'GPUs: {tf.config.list_physical_devices("GPU")}')

TensorFlow 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


## 2 — Configuration

In [4]:
# ── Dataset ───────────────────────────────────────────────────────────────────
DATA_DIR = r"/kaggle/input/datasets/rracer17/mit-bih-mitdb/mit-bih-arrhythmia-database-1.0.0"

# ── Paper constants ──────────────────────────────────────────────────────────
FS           = 360
TOTAL_STEPS  = 100_000
TRAIN_STEPS  = 40_000
VAL_STEPS    = 10_000
TEST_STEPS   = 50_000
LOOKBACK     = 10

# ── Patients ─────────────────────────────────────────────────────────────────
PATIENTS = [
    '100', '101', '102', '103', '104', '105',
    '106', '107', '108', '109', '112', '113',
    '114', '115', '116', '117', '119',
    '121', '122', '123', '124'
]
assert len(PATIENTS) == 21

# ── Single horizon ───────────────────────────────────────────────────────────
HORIZON = 10

# ┌──────────────────────────────────────────────────────────────────────────┐
# │  GWO-OPTIMIZED HYPERPARAMETERS                                         │
# │  Source: GWO search Eval 5, pilot RMSE = 0.026152                      │
# └──────────────────────────────────────────────────────────────────────────┘
NUM_FILTERS   = 256
DROPOUT_RATE  = 0.109
LEARNING_RATE = 0.002066
NUM_BLOCKS    = 4
KERNEL_SIZE   = 3
BATCH_SIZE    = 128

# ── Training ─────────────────────────────────────────────────────────────────
EPOCHS   = 200
PATIENCE = 50

# ── Output ───────────────────────────────────────────────────────────────────
CONFIG_NAME = 'GWO_Optimized'
CSV_NAME    = 'tcn_mo_gwo_optimized_results.csv'
PLOT_DIR    = 'plots_gwo_optimized'
os.makedirs(PLOT_DIR, exist_ok=True)

print(f'Config    : {CONFIG_NAME}')
print(f'Patients  : {len(PATIENTS)}')
print(f'Horizon   : H={HORIZON}')
print(f'TCN       : blocks={NUM_BLOCKS}, filters={NUM_FILTERS}, kernel={KERNEL_SIZE}')
print(f'Dropout   : {DROPOUT_RATE}')
print(f'Training  : epochs={EPOCHS}, patience={PATIENCE}, batch={BATCH_SIZE}, lr={LEARNING_RATE:.6f}')

Config    : GWO_Optimized
Patients  : 21
Horizon   : H=10
TCN       : blocks=4, filters=256, kernel=3
Dropout   : 0.109
Training  : epochs=200, patience=50, batch=128, lr=0.002066


## 3 — Data Pipeline

In [5]:
def load_ecg_signal(record_id, data_dir, n_steps=100_000):
    """Load MLII lead from MIT-BIH, truncate/pad to n_steps."""
    path = os.path.join(data_dir, record_id)
    rec  = wfdb.rdrecord(path)
    sig_names_upper = [s.upper() for s in rec.sig_name]
    ch = sig_names_upper.index('MLII') if 'MLII' in sig_names_upper else 0
    signal = rec.p_signal[:, ch].astype(np.float32)
    if len(signal) < n_steps:
        pad = np.full(n_steps - len(signal), signal[-1], dtype=np.float32)
        signal = np.concatenate([signal, pad])
    return signal[:n_steps]


def preprocess_patient(signal, train_steps=40_000, val_steps=10_000):
    """Split → train/val/test, MinMax-normalise (fit on train only)."""
    train_end = train_steps
    val_end   = train_steps + val_steps
    train_raw = signal[:train_end]
    val_raw   = signal[train_end:val_end]
    test_raw  = signal[val_end:]
    scaler     = MinMaxScaler(feature_range=(0, 1))
    train_norm = scaler.fit_transform(train_raw.reshape(-1, 1)).flatten()
    val_norm   = scaler.transform(val_raw.reshape(-1, 1)).flatten()
    test_norm  = scaler.transform(test_raw.reshape(-1, 1)).flatten()
    return train_norm, val_norm, test_norm, scaler


def make_multistep_sequences(signal, lookback, horizon):
    """X = lookback window, y = next H steps (Multi-Output)."""
    X, y = [], []
    for i in range(len(signal) - lookback - horizon + 1):
        X.append(signal[i : i + lookback])
        y.append(signal[i + lookback : i + lookback + horizon])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


def compute_metrics(y_true, y_pred):
    yt, yp = y_true.flatten(), y_pred.flatten()
    return {
        'RMSE': float(np.sqrt(mean_squared_error(yt, yp))),
        'MAE':  float(mean_absolute_error(yt, yp)),
        'R2':   float(r2_score(yt, yp)),
    }


def compute_per_step_metrics(y_true, y_pred):
    rows = []
    for h in range(y_true.shape[1]):
        m = compute_metrics(y_true[:, h], y_pred[:, h])
        m['Step'] = h + 1
        rows.append(m)
    return pd.DataFrame(rows)


print('Data pipeline defined.')

Data pipeline defined.


In [6]:
# ── Load all patients ─────────────────────────────────────────────────────────
patient_signals = {}

print(f'Loading {len(PATIENTS)} patients...\n')
for rid in tqdm(PATIENTS, desc='Patients'):
    signal = load_ecg_signal(rid, DATA_DIR, n_steps=TOTAL_STEPS)
    tr, vl, te, sc = preprocess_patient(signal, TRAIN_STEPS, VAL_STEPS)
    patient_signals[rid] = {'train': tr, 'val': vl, 'test': te, 'scaler': sc}
    tqdm.write(f'  Patient {rid:>3s} | train={len(tr):,}  val={len(vl):,}  test={len(te):,}')

print(f'\nAll {len(patient_signals)} patients loaded.')

Loading 21 patients...



Patients:   0%|          | 0/21 [00:00<?, ?it/s]

  Patient 100 | train=40,000  val=10,000  test=50,000
  Patient 101 | train=40,000  val=10,000  test=50,000
  Patient 102 | train=40,000  val=10,000  test=50,000
  Patient 103 | train=40,000  val=10,000  test=50,000
  Patient 104 | train=40,000  val=10,000  test=50,000
  Patient 105 | train=40,000  val=10,000  test=50,000
  Patient 106 | train=40,000  val=10,000  test=50,000
  Patient 107 | train=40,000  val=10,000  test=50,000
  Patient 108 | train=40,000  val=10,000  test=50,000
  Patient 109 | train=40,000  val=10,000  test=50,000
  Patient 112 | train=40,000  val=10,000  test=50,000
  Patient 113 | train=40,000  val=10,000  test=50,000
  Patient 114 | train=40,000  val=10,000  test=50,000
  Patient 115 | train=40,000  val=10,000  test=50,000
  Patient 116 | train=40,000  val=10,000  test=50,000
  Patient 117 | train=40,000  val=10,000  test=50,000
  Patient 119 | train=40,000  val=10,000  test=50,000
  Patient 121 | train=40,000  val=10,000  test=50,000
  Patient 122 | train=40,000

## 4 — Model Architecture

In [7]:
def residual_block(x, filters, kernel_size, dilation_rate, dropout_rate):
    out = Conv1D(filters, kernel_size, dilation_rate=dilation_rate, padding='causal')(x)
    out = BatchNormalization()(out)
    out = Activation('relu')(out)
    out = Dropout(dropout_rate)(out)
    out = Conv1D(filters, kernel_size, dilation_rate=dilation_rate, padding='causal')(out)
    out = BatchNormalization()(out)
    out = Activation('relu')(out)
    out = Dropout(dropout_rate)(out)
    if x.shape[-1] != filters:
        x = Conv1D(filters, 1)(x)
    return Add()([x, out])


def build_tcn(lookback, output_size, n_blocks, n_filters, kernel_size, dropout_rate, learning_rate):
    inp = Input(shape=(lookback, 1))
    x = inp
    for i in range(n_blocks):
        x = residual_block(x, n_filters, kernel_size, 2 ** i, dropout_rate)
    x = x[:, -1, :]
    x = Dense(n_filters, activation='relu')(x)
    out = Dense(output_size)(x)
    model = Model(inp, out, name=f'TCN_H{output_size}')
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate), loss='mse')
    return model


# Show model summary
demo_model = build_tcn(LOOKBACK, HORIZON, NUM_BLOCKS, NUM_FILTERS, KERNEL_SIZE, DROPOUT_RATE, LEARNING_RATE)
demo_model.summary()
del demo_model

I0000 00:00:1786806013.272920      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786806013.275791      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "TCN_H10"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 10, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 10, 256)   │      1,024 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 10, 256)   │      1,024 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 10, 256)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 10, 256)   │          0 │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 10, 256)   │    196,864 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 10, 256)   │      1,024 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 10, 256)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 10, 256)   │        512 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 10, 256)   │          0 │ activation_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 10, 256)   │          0 │ conv1d_2[0][0],   │
│                     │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 10, 256)   │    196,864 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 10, 256)   │      1,024 │ conv1d_3[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 10, 256)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 10, 256)   │          0 │ activation_2[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 10, 256)   │    196,864 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 10, 256)   │      1,024 │ conv1d_4[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 10, 256)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 10, 256)   │          0 │ activation_3[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 10, 256)   │          0 │ add[0][0],      

 Total params: 1,456,138 (5.55 MB)

 Trainable params: 1,452,042 (5.54 MB)

 Non-trainable params: 4,096 (16.00 KB)

## 5 — Train & Evaluate All Patients

In [8]:
results            = []
per_step_results   = {}
saved_preds        = {}
training_histories = {}

print(f'Running: {len(PATIENTS)} patients × H={HORIZON}')
print(f'Config : blocks={NUM_BLOCKS}, filters={NUM_FILTERS}, kernel={KERNEL_SIZE}, '
      f'dropout={DROPOUT_RATE}, lr={LEARNING_RATE}, batch={BATCH_SIZE}')
print('=' * 80)

for rid in tqdm(PATIENTS, desc='Patients'):
    tf.keras.backend.clear_session()
    gc.collect()

    tr = patient_signals[rid]['train']
    vl = patient_signals[rid]['val']
    te = patient_signals[rid]['test']

    X_tr, y_tr = make_multistep_sequences(tr, LOOKBACK, HORIZON)
    X_vl, y_vl = make_multistep_sequences(vl, LOOKBACK, HORIZON)
    X_te, y_te = make_multistep_sequences(te, LOOKBACK, HORIZON)

    X_tr_r = X_tr.reshape(-1, X_tr.shape[1], 1)
    X_vl_r = X_vl.reshape(-1, X_vl.shape[1], 1)
    X_te_r = X_te.reshape(-1, X_te.shape[1], 1)

    t0 = time.time()
    model = build_tcn(LOOKBACK, HORIZON,
                      NUM_BLOCKS, NUM_FILTERS, KERNEL_SIZE,
                      DROPOUT_RATE, LEARNING_RATE)

    history = model.fit(
        X_tr_r, y_tr,
        validation_data=(X_vl_r, y_vl),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[
            EarlyStopping(monitor='val_loss', patience=PATIENCE,
                          restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                              patience=25, min_lr=1e-6),
        ],
        verbose=0
    )

    preds = model.predict(X_te_r, batch_size=2048, verbose=0)
    elapsed = time.time() - t0

    m = compute_metrics(y_te, preds)
    m['Time_s'] = round(elapsed, 2)
    m['Epochs_trained'] = len(history.history['loss'])
    results.append({'Patient': rid, 'Horizon': HORIZON, **m})

    per_step_results[rid] = compute_per_step_metrics(y_te, preds)
    saved_preds[rid] = (y_te.copy(), preds.copy())
    training_histories[rid] = {
        'loss': history.history['loss'],
        'val_loss': history.history['val_loss'],
    }

    del model
    gc.collect()

    tqdm.write(
        f'  Patient {rid} | R²={m["R2"]:.4f}  RMSE={m["RMSE"]:.4f}  '
        f'MAE={m["MAE"]:.4f}  ({m["Epochs_trained"]} ep, {elapsed:.1f}s)'
    )

print(f'\nAll {len(results)} patients complete.')

Running: 21 patients × H=10
Config : blocks=4, filters=256, kernel=3, dropout=0.109, lr=0.002066, batch=128


Patients:   0%|          | 0/21 [00:00<?, ?it/s]

I0000 00:00:1786806030.371118      76 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2026-08-15 15:07:05.217876: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng18{k11=0} for conv (f32[256,16384,1,2]{3,2,1,0}, u8[0]{0}) custom-call(f32[256,16384,1,4]{3,2,1,0}, f32[256,256,1,3]{3,2,1,0}), window={size=1x2}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBackwardFilter", backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]} is taking a while...
2026-08-15 15:07:05.897777: E external/local_xla/xla/service/slow_operation_alarm.cc:140] The operation took 1.680022788s
Trying algorithm eng18{k11=0} for conv (f32[256,16384,1,2]{3,2,1,0}, u8[0]{0}) custom-call(f32[256,16384,1,4]{3,2,1,0}, f32

  Patient 100 | R²=0.9262  RMSE=0.0261  MAE=0.0129  (122 ep, 414.6s)
  Patient 101 | R²=0.6620  RMSE=0.0639  MAE=0.0217  (68 ep, 236.8s)
  Patient 102 | R²=0.8208  RMSE=0.0286  MAE=0.0112  (167 ep, 541.2s)
  Patient 103 | R²=0.9783  RMSE=0.0171  MAE=0.0081  (200 ep, 654.6s)
  Patient 104 | R²=0.7578  RMSE=0.0513  MAE=0.0273  (125 ep, 413.7s)
  Patient 105 | R²=0.8938  RMSE=0.0376  MAE=0.0156  (192 ep, 626.2s)
  Patient 106 | R²=0.8677  RMSE=0.0339  MAE=0.0121  (165 ep, 543.5s)
  Patient 107 | R²=0.9476  RMSE=0.0301  MAE=0.0126  (184 ep, 594.7s)
  Patient 108 | R²=0.7619  RMSE=0.0443  MAE=0.0230  (118 ep, 406.0s)
  Patient 109 | R²=0.9872  RMSE=0.0156  MAE=0.0084  (200 ep, 707.4s)
  Patient 112 | R²=0.8932  RMSE=0.0375  MAE=0.0177  (200 ep, 706.5s)
  Patient 113 | R²=0.9758  RMSE=0.0201  MAE=0.0074  (200 ep, 698.6s)
  Patient 114 | R²=0.9606  RMSE=0.0129  MAE=0.0051  (180 ep, 635.4s)
  Patient 115 | R²=0.8893  RMSE=0.0281  MAE=0.0093  (200 ep, 701.6s)
  Patient 116 | R²=0.9300  RMSE=0.0

In [9]:
# ── Save results ─────────────────────────────────────────────────────────────
df_results = pd.DataFrame(results)
df_results.to_csv(CSV_NAME, index=False)
print(f'Results saved to {CSV_NAME}\n')
display(df_results)

print(f'\n--- Summary (H={HORIZON}) ---')
print(f'Mean RMSE : {df_results.RMSE.mean():.6f} ± {df_results.RMSE.std():.6f}')
print(f'Mean MAE  : {df_results.MAE.mean():.6f} ± {df_results.MAE.std():.6f}')
print(f'Mean R²   : {df_results.R2.mean():.4f} ± {df_results.R2.std():.4f}')

Results saved to tcn_mo_gwo_optimized_results.csv



,Patient,Horizon,RMSE,MAE,R2,Time_s,Epochs_trained
0,100,10,0.026111,0.012896,0.926181,414.57,122
1,101,10,0.063896,0.021736,0.662033,236.79,68
2,102,10,0.028617,0.011196,0.820800,541.23,167
3,103,10,0.017083,0.008118,0.978319,654.57,200
4,104,10,0.051275,0.027296,0.757826,413.65,125
5,105,10,0.037569,0.015648,0.893831,626.18,192
6,106,10,0.033945,0.012149,0.867732,543.48,165
7,107,10,0.030133,0.012627,0.947614,594.70,184
8,108,10,0.044255,0.022964,0.761901,406.03,118
9,109,10,0.015569,0.008351,0.987155,707.40,200



--- Summary (H=10) ---
Mean RMSE : 0.029703 ± 0.012353
Mean MAE  : 0.012890 ± 0.005637
Mean R²   : 0.9020 ± 0.0865


## 6 — Comparison: Baseline vs GWO-Optimized

In [10]:
# ── Load baseline ────────────────────────────────────────────────────────────
df_baseline = pd.read_csv('Figures/results/tcn_mo_ours_config_results.csv')
df_base_h10 = df_baseline[df_baseline.Horizon == HORIZON].sort_values('Patient').reset_index(drop=True)
df_opt_h10  = df_results.sort_values('Patient').reset_index(drop=True)

rmse_b, rmse_o = df_base_h10.RMSE.mean(), df_opt_h10.RMSE.mean()
mae_b, mae_o   = df_base_h10.MAE.mean(), df_opt_h10.MAE.mean()
r2_b, r2_o     = df_base_h10.R2.mean(), df_opt_h10.R2.mean()

print('Baseline (Ours) vs GWO-Optimized — H=10')
print('=' * 60)
print(f'{"Metric":<8s} {"Baseline":>12s} {"GWO-Opt":>12s} {"Δ%":>8s}')
print('-' * 60)
print(f'{"RMSE":<8s} {rmse_b:12.6f} {rmse_o:12.6f} {(rmse_o-rmse_b)/rmse_b*100:+7.1f}%')
print(f'{"MAE":<8s} {mae_b:12.6f} {mae_o:12.6f} {(mae_o-mae_b)/mae_b*100:+7.1f}%')
print(f'{"R²":<8s} {r2_b:12.4f} {r2_o:12.4f} {(r2_o-r2_b)/r2_b*100:+7.2f}%')
print('=' * 60)

wins = (df_opt_h10.RMSE.values < df_base_h10.RMSE.values).sum()
print(f'\nGWO-Optimized wins {wins}/{len(df_base_h10)} patients ({wins/len(df_base_h10)*100:.0f}%)')

FileNotFoundError: [Errno 2] No such file or directory: 'Figures/results/tcn_mo_ours_config_results.csv'

In [ ]:
# ── Wilcoxon signed-rank test ────────────────────────────────────────────────
from scipy.stats import wilcoxon

stat, p = wilcoxon(df_base_h10.RMSE.values, df_opt_h10.RMSE.values)
sig = '✓ Significant' if p < 0.05 else '✗ Not significant'
print(f'Wilcoxon signed-rank test (H={HORIZON}): p={p:.4f} → {sig}')

## 7 — Visualizations

In [ ]:
# ── 7.1 Per-Patient RMSE Bar Chart (Baseline vs GWO) ─────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))
x = np.arange(len(df_base_h10))
w = 0.35

ax.bar(x - w/2, df_base_h10.RMSE.values, w,
       label='Baseline (Ours)', color='#90CAF9', edgecolor='white')
ax.bar(x + w/2, df_opt_h10.RMSE.values, w,
       label='GWO-Optimized', color='#1565C0', edgecolor='white')

ax.set_xlabel('Patient', fontsize=12)
ax.set_ylabel('RMSE', fontsize=12)
ax.set_title(f'Per-Patient RMSE — Baseline vs GWO-Optimized (H={HORIZON})', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(df_base_h10.Patient.values, rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/per_patient_rmse_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7.2 Per-Patient R² Bar Chart ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))

ax.bar(x - w/2, df_base_h10.R2.values, w,
       label='Baseline (Ours)', color='#A5D6A7', edgecolor='white')
ax.bar(x + w/2, df_opt_h10.R2.values, w,
       label='GWO-Optimized', color='#2E7D32', edgecolor='white')

ax.set_xlabel('Patient', fontsize=12)
ax.set_ylabel('R²', fontsize=12)
ax.set_title(f'Per-Patient R² — Baseline vs GWO-Optimized (H={HORIZON})', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(df_base_h10.Patient.values, rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/per_patient_r2_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7.3 RMSE Improvement per Patient (waterfall) ─────────────────────────────
improvement = df_base_h10.RMSE.values - df_opt_h10.RMSE.values  # positive = GWO better
colors = ['#2E7D32' if imp > 0 else '#C62828' for imp in improvement]

fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(x, improvement, color=colors, edgecolor='white')
ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_xlabel('Patient', fontsize=12)
ax.set_ylabel('RMSE Improvement (Baseline − GWO)', fontsize=12)
ax.set_title(f'RMSE Improvement per Patient (H={HORIZON}) — Green = GWO Better', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(df_base_h10.Patient.values, rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/rmse_improvement_waterfall.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7.4 Training Curves (loss & val_loss) for all patients ───────────────────
n_cols = 7
n_rows = 3
fig, axes = plt.subplots(n_rows, n_cols, figsize=(28, 10), sharey=False)

for idx, rid in enumerate(PATIENTS):
    row, col = idx // n_cols, idx % n_cols
    ax = axes[row, col]
    hist = training_histories[rid]
    epochs_range = range(1, len(hist['loss']) + 1)
    ax.plot(epochs_range, hist['loss'], label='Train', linewidth=0.8, color='#1565C0')
    ax.plot(epochs_range, hist['val_loss'], label='Val', linewidth=0.8, color='#E53935')
    ax.set_title(f'Patient {rid}', fontsize=10)
    ax.set_xlabel('Epoch', fontsize=8)
    if col == 0:
        ax.set_ylabel('Loss (MSE)', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(alpha=0.2)
    if idx == 0:
        ax.legend(fontsize=7)

plt.suptitle(f'Training Curves — GWO-Optimized TCN (H={HORIZON})', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/training_curves_all.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7.5 Per-Step RMSE Degradation ────────────────────────────────────────────
# Show how RMSE grows across forecast steps 1→10
all_step_rmse = np.zeros(HORIZON)
for rid in PATIENTS:
    df_steps = per_step_results[rid]
    all_step_rmse += df_steps.RMSE.values
all_step_rmse /= len(PATIENTS)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, HORIZON + 1), all_step_rmse, 'o-', color='#1565C0',
        linewidth=2, markersize=8)
ax.set_xlabel('Forecast Step', fontsize=12)
ax.set_ylabel('Mean RMSE', fontsize=12)
ax.set_title(f'RMSE Degradation Across Forecast Steps (H={HORIZON}, avg over {len(PATIENTS)} patients)',
             fontsize=13)
ax.set_xticks(range(1, HORIZON + 1))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/per_step_rmse_degradation.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7.6 Prediction Samples (4 representative patients) ──────────────────────
sample_patients = ['100', '103', '108', '124']
n_show = 500  # timesteps to show

fig, axes = plt.subplots(len(sample_patients), 1, figsize=(16, 3.5 * len(sample_patients)))

for ax, rid in zip(axes, sample_patients):
    y_true, y_pred = saved_preds[rid]
    # Show step 1 predictions vs actual (most visible)
    actual  = y_true[:n_show, 0]
    predicted = y_pred[:n_show, 0]

    ax.plot(actual, label='Actual', color='#333333', linewidth=0.8, alpha=0.8)
    ax.plot(predicted, label='Predicted (step 1)', color='#1565C0', linewidth=0.8, alpha=0.8)
    rmse_p = float(np.sqrt(mean_squared_error(actual, predicted)))
    ax.set_title(f'Patient {rid} — Step 1 Prediction (RMSE={rmse_p:.4f})', fontsize=11)
    ax.set_xlabel('Time Index')
    ax.set_ylabel('Normalised ECG')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(alpha=0.2)

plt.suptitle(f'Prediction Samples — GWO-Optimized TCN (H={HORIZON})', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/prediction_samples.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7.7 Multi-Step Prediction Fan (show all 10 steps for 1 patient) ──────────
rid = '100'
y_true, y_pred = saved_preds[rid]
start = 200  # starting index
window = 100

fig, ax = plt.subplots(figsize=(16, 5))

# Plot actual for all steps
for step in range(HORIZON):
    alpha = 1.0 - step * 0.08
    if step == 0:
        ax.plot(range(start, start + window),
                y_true[start:start+window, step],
                color='#333333', linewidth=1.2, alpha=0.9, label='Actual')
        ax.plot(range(start, start + window),
                y_pred[start:start+window, step],
                color='#1565C0', linewidth=1.2, alpha=0.9, label='Pred step 1')
    elif step == HORIZON - 1:
        ax.plot(range(start, start + window),
                y_true[start:start+window, step],
                color='#333333', linewidth=0.5, alpha=0.4)
        ax.plot(range(start, start + window),
                y_pred[start:start+window, step],
                color='#E53935', linewidth=1.0, alpha=0.7, label=f'Pred step {HORIZON}')

ax.set_xlabel('Time Index', fontsize=12)
ax.set_ylabel('Normalised ECG', fontsize=12)
ax.set_title(f'Patient {rid} — Step 1 vs Step {HORIZON} Predictions', fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/multistep_fan_{rid}.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7.8 Metrics Distribution (Box Plots) ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric, title in zip(axes,
    ['RMSE', 'MAE', 'R2'],
    ['RMSE ↓', 'MAE ↓', 'R² ↑']):

    data = [df_base_h10[metric].values, df_opt_h10[metric].values]
    bp = ax.boxplot(data, labels=['Baseline', 'GWO-Opt'], patch_artist=True,
                    widths=0.5)
    bp['boxes'][0].set_facecolor('#90CAF9')
    bp['boxes'][1].set_facecolor('#1565C0')
    bp['boxes'][1].set(alpha=0.8)
    ax.set_ylabel(metric, fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle(f'Metrics Distribution — Baseline vs GWO-Optimized (H={HORIZON})', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/metrics_boxplots.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7.9 Per-Step Metrics Heatmap (all patients × 10 steps) ───────────────────
step_rmse_matrix = np.zeros((len(PATIENTS), HORIZON))
for i, rid in enumerate(PATIENTS):
    step_rmse_matrix[i, :] = per_step_results[rid].RMSE.values

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(step_rmse_matrix, annot=True, fmt='.4f', cmap='YlOrRd',
            xticklabels=[f'Step {i+1}' for i in range(HORIZON)],
            yticklabels=PATIENTS, ax=ax, linewidths=0.5)
ax.set_xlabel('Forecast Step', fontsize=12)
ax.set_ylabel('Patient', fontsize=12)
ax.set_title(f'Per-Step RMSE Heatmap — GWO-Optimized (H={HORIZON})', fontsize=14)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/per_step_rmse_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

## 8 — Summary

In [ ]:
print('\n' + '=' * 80)
print('GWO-OPTIMIZED TCN — FINAL EVALUATION COMPLETE')
print('=' * 80)

print(f'\nConfiguration:')
print(f'  n_filters     = {NUM_FILTERS}')
print(f'  dropout       = {DROPOUT_RATE}')
print(f'  lr            = {LEARNING_RATE}')
print(f'  n_blocks      = {NUM_BLOCKS} (fixed)')
print(f'  kernel_size   = {KERNEL_SIZE} (fixed)')
print(f'  batch_size    = {BATCH_SIZE} (fixed)')

print(f'\nResults (H={HORIZON}, {len(PATIENTS)} patients):')
print(f'  Mean RMSE     = {df_results.RMSE.mean():.6f} ± {df_results.RMSE.std():.6f}')
print(f'  Mean MAE      = {df_results.MAE.mean():.6f} ± {df_results.MAE.std():.6f}')
print(f'  Mean R²       = {df_results.R2.mean():.4f} ± {df_results.R2.std():.4f}')

print(f'\nvs Baseline:')
rmse_pct = (rmse_o - rmse_b) / rmse_b * 100
r2_pct   = (r2_o - r2_b) / r2_b * 100
print(f'  RMSE  : {rmse_b:.6f} → {rmse_o:.6f} ({rmse_pct:+.1f}%)')
print(f'  R²    : {r2_b:.4f} → {r2_o:.4f} ({r2_pct:+.2f}%)')
print(f'  Wins  : {wins}/{len(df_base_h10)} patients')

print(f'\nOutputs:')
print(f'  CSV   : {CSV_NAME}')
print(f'  Plots : {PLOT_DIR}/')
print('=' * 80)